# Day 3 — SQL Practice Questions: Date & Time Functions

| # | Difficulty | Topic |
|---|-----------|-------|
| 1 | 🟢 Easy    | Find slow deliveries (date arithmetic) |
| 2 | 🟢 Easy    | Monthly revenue totals (DATE_TRUNC + GROUP BY) |
| 3 | 🟡 Medium  | New customers — first order in last 90 days (MIN + HAVING) |

In [ ]:
%load_ext sql
USERNAME = 'postgres'
PASSWORD = 'hariom'
HOST     = 'localhost'
PORT     = '5432'
DBNAME   = 'postgres'
%sql postgresql://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}

---
# Question 1 🟢 — Slow Deliveries

## Concept Warm-up

In [ ]:
%%sql
-- Warm-up 1: subtract two dates → integer days in PostgreSQL
SELECT '2026-05-14'::date - '2026-05-03'::date AS days_diff;

In [ ]:
%%sql
-- Warm-up 2: add interval to a date
SELECT '2026-05-01'::date + INTERVAL '7 days'  AS plus_7,
       '2026-05-01'::date + INTERVAL '30 days' AS plus_30,
       '2026-05-01'::date + 14                  AS plus_14_shorthand;

In [ ]:
%%sql
-- Warm-up 3: preview the pq_orders table
SELECT order_id, customer_id, order_date, ship_date, amount
FROM   pq_orders
ORDER  BY order_date
LIMIT  8;

In [ ]:
%%sql
-- Warm-up 4: compute delivery_days for all rows
SELECT order_id,
       order_date,
       ship_date,
       ship_date - order_date AS delivery_days
FROM   pq_orders
ORDER  BY delivery_days DESC
LIMIT  5;

## Problem

The logistics team wants to review orders where the delivery took **more than 7 days**.  
Return `order_id`, `customer_id`, `order_date`, `ship_date`, and `delivery_days`, sorted by slowest delivery first.

```
Expected: rows where ship_date - order_date > 7
```

In [ ]:
%%sql
-- YOUR QUERY HERE

---
# Question 2 🟢 — Monthly Revenue Totals

## Concept Warm-up

In [ ]:
%%sql
-- Warm-up 1: DATE_TRUNC maps every date to the first of its month
SELECT order_date,
       DATE_TRUNC('month', order_date) AS month_start
FROM   pq_orders
LIMIT  6;

In [ ]:
%%sql
-- Warm-up 2: TO_CHAR formats a date as a readable label
SELECT order_date,
       TO_CHAR(order_date, 'YYYY-MM')  AS ym,
       TO_CHAR(order_date, 'Mon YYYY') AS label
FROM   pq_orders
LIMIT  5;

In [ ]:
%%sql
-- Warm-up 3: GROUP BY DATE_TRUNC → chronological aggregation
SELECT DATE_TRUNC('month', order_date) AS month,
       COUNT(*) AS num_orders
FROM   pq_orders
GROUP  BY 1
ORDER  BY 1;

## Problem

The finance team needs a **monthly revenue report**: total revenue, number of orders, and average order value per month, ordered from oldest to most recent.  
Display the month as `'YYYY-MM'` string in the output.

```
Columns: month (YYYY-MM), total_revenue, num_orders, avg_order_value
```

In [ ]:
%%sql
-- YOUR QUERY HERE

---
# Question 3 🟡 — New Customers (First Order in Last 90 Days)

## Concept Warm-up

In [ ]:
%%sql
-- Warm-up 1: CURRENT_DATE and 90-day window
SELECT CURRENT_DATE                            AS today,
       CURRENT_DATE - INTERVAL '90 days'       AS ninety_days_ago,
       CURRENT_DATE - 90                       AS same_shorthand;

In [ ]:
%%sql
-- Warm-up 2: MIN(order_date) per customer — first order date
SELECT customer_id,
       MIN(order_date) AS first_order_date,
       COUNT(*)        AS total_orders
FROM   pq_orders
GROUP  BY customer_id
ORDER  BY first_order_date;

In [ ]:
%%sql
-- Warm-up 3: HAVING filters on aggregates — WHERE cannot
-- Show customers with more than 2 orders
SELECT customer_id, COUNT(*) AS num_orders
FROM   pq_orders
GROUP  BY customer_id
HAVING COUNT(*) > 2
ORDER  BY num_orders DESC;

In [ ]:
%%sql
-- Warm-up 4: CURRENT_DATE - MIN(order_date) → days since first order
SELECT customer_id,
       MIN(order_date)                          AS first_order_date,
       CURRENT_DATE - MIN(order_date)           AS days_since_first
FROM   pq_orders
GROUP  BY customer_id
ORDER  BY days_since_first ASC;

## Problem

The growth team defines a **new customer** as someone whose **first-ever order** was placed within the last 90 days.  
Return `customer_id`, `first_order_date`, and `days_since_first_order`, sorted by most-recent first.

**Hint:** Use `GROUP BY` + `MIN(order_date)` + `HAVING`.

In [ ]:
%%sql
-- YOUR QUERY HERE